# Correlation Study: AI Vibrancy, Economic Freedom, and Regulatory Framing

This notebook tests the core relationship between Stanford AI Vibrancy, Heritage Index of Economic Freedom (IEF), and AI regulatory framing for G7 + China.

The default framing measure is the Qwen sentence-level classifier output. The older semantic keyword/rule framing output can still be used as a robustness check by changing `FRAMING_SOURCE` below.

## Hypotheses

**H1 — AI Vibrancy and Innovation-Oriented Regulatory Framing**  
Countries with higher Stanford AI Vibrancy scores will exhibit higher innovation-to-risk framing ratios.

**H2 — Economic Freedom and Positive Regulatory Framing**  
Countries whose AI regulatory discourse is more positive and innovation-oriented will exhibit higher economic freedom scores.

**H3 — Regulatory Framing as a Mediating Mechanism**  
The relationship between AI vibrancy and economic freedom is partially or fully mediated by the permissiveness of AI regulatory rhetoric.

Because this is an eight-country cross-sectional study, all inferential statistics should be interpreted as exploratory.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

outputs = repo_root / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)

G7_CHINA_COUNTRIES = [
    'United States',
    'China',
    'United Kingdom',
    'France',
    'Germany',
    'Italy',
    'Japan',
    'Canada',
]

# Use 'qwen' for the Qwen classifier output. Use 'semantic' for the older keyword/rule robustness check.
FRAMING_SOURCE = 'qwen'
EPSILON = 0.1

repo_root

## Load and Merge Data

In [ ]:
def load_vibrancy():
    path = repo_root / 'data' / 'AI vibrancy tool screen shot' / 'ai_country_scores.csv'
    df = pd.read_csv(path)
    return df.rename(columns={'Total Score': 'AI Vibrancy Score'})


def load_latest_ief():
    path = repo_root / 'data' / 'number' / 'ief_g7_china_panel.csv'
    df = pd.read_csv(path)
    latest_year = int(df['Index Year'].max())
    latest = df[df['Index Year'] == latest_year].copy()
    latest = latest.rename(columns={'Overall Score': 'IEF Overall Score'})
    return latest[['Country', 'Index Year', 'IEF Overall Score']]


def load_framing(source='semantic'):
    if source == 'semantic':
        path = outputs / 'semantic_ai_framing_summary.csv'
        framing_label = 'Keyword framing'
    elif source == 'qwen':
        path = outputs / 'qwen_ai_framing_summary.csv'
        framing_label = 'Qwen framing'
    else:
        raise ValueError("source must be 'semantic' or 'qwen'")

    df = pd.read_csv(path)
    df = df.rename(columns={
        'AI-friendly per 1000 words': 'Innovation Framing per 1000 words',
        'AI-cautious per 1000 words': 'Risk Framing per 1000 words',
        'Net AI-friendly framing': 'Net Innovation Framing',
        'AI-friendly share': 'Innovation Framing Share',
    })
    df['Framing Source'] = framing_label
    df['Innovation to Risk Ratio'] = (
        (df['Innovation Framing per 1000 words'] + EPSILON)
        / (df['Risk Framing per 1000 words'] + EPSILON)
    )
    df['Log Innovation to Risk Ratio'] = np.log(df['Innovation to Risk Ratio'])
    return df[[
        'Country', 'Framing Source', 'Word count',
        'Innovation Framing per 1000 words', 'Risk Framing per 1000 words',
        'Innovation to Risk Ratio', 'Log Innovation to Risk Ratio',
        'Net Innovation Framing', 'Innovation Framing Share',
    ]]


vibrancy = load_vibrancy()
ief = load_latest_ief()
framing = load_framing(FRAMING_SOURCE)

study = (
    pd.DataFrame({'Country': G7_CHINA_COUNTRIES})
    .merge(vibrancy, on='Country', how='left')
    .merge(ief, on='Country', how='left')
    .merge(framing, on='Country', how='left')
)

study_path = outputs / f'correlation_study_dataset_{FRAMING_SOURCE}.csv'
study.to_csv(study_path, index=False)
print(f'Saved: {study_path}')
study

## Correlation Tests

In [ ]:
def correlation_row(df, x, y, hypothesis):
    clean = df[[x, y]].dropna()
    pearson_r, pearson_p = stats.pearsonr(clean[x], clean[y])
    spearman_r, spearman_p = stats.spearmanr(clean[x], clean[y])
    return {
        'Hypothesis': hypothesis,
        'X': x,
        'Y': y,
        'N': len(clean),
        'Pearson r': round(float(pearson_r), 3),
        'Pearson p': round(float(pearson_p), 4),
        'Spearman rho': round(float(spearman_r), 3),
        'Spearman p': round(float(spearman_p), 4),
    }


corr_rows = [
    correlation_row(study, 'AI Vibrancy Score', 'Log Innovation to Risk Ratio', 'H1'),
    correlation_row(study, 'IEF Overall Score', 'Log Innovation to Risk Ratio', 'H2'),
    correlation_row(study, 'AI Vibrancy Score', 'IEF Overall Score', 'H3 total relationship'),
]
correlations = pd.DataFrame(corr_rows)

corr_path = outputs / f'correlation_study_correlations_{FRAMING_SOURCE}.csv'
correlations.to_csv(corr_path, index=False)
print(f'Saved: {corr_path}')
correlations

## OLS Models

The combined model operationalizes the requested comparison: **AI Vibrancy + IEF combined vs. Framing Keyword Ratio**.

In [ ]:
def fit_ols(df, y_col, x_cols, model_name):
    clean = df[[y_col, *x_cols]].dropna().copy()
    y = clean[y_col].to_numpy(dtype=float)
    x = clean[x_cols].to_numpy(dtype=float)
    x_design = np.column_stack([np.ones(len(clean)), x])
    beta, *_ = np.linalg.lstsq(x_design, y, rcond=None)
    fitted = x_design @ beta
    resid = y - fitted
    n = len(y)
    k = x_design.shape[1]
    sse = float(np.sum(resid ** 2))
    sst = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - sse / sst if sst else np.nan
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - k) if n > k and not np.isnan(r2) else np.nan
    sigma2 = sse / (n - k) if n > k else np.nan
    cov = sigma2 * np.linalg.pinv(x_design.T @ x_design) if n > k else np.full((k, k), np.nan)
    se = np.sqrt(np.diag(cov))
    t_values = beta / se
    p_values = 2 * stats.t.sf(np.abs(t_values), df=n - k) if n > k else np.full(k, np.nan)

    names = ['Intercept', *x_cols]
    rows = []
    for name, coef, se_i, t_i, p_i in zip(names, beta, se, t_values, p_values):
        rows.append({
            'Model': model_name,
            'Outcome': y_col,
            'Term': name,
            'Coefficient': round(float(coef), 4),
            'Std. Error': round(float(se_i), 4) if np.isfinite(se_i) else np.nan,
            't': round(float(t_i), 3) if np.isfinite(t_i) else np.nan,
            'p': round(float(p_i), 4) if np.isfinite(p_i) else np.nan,
            'N': n,
            'R2': round(float(r2), 3) if np.isfinite(r2) else np.nan,
            'Adj. R2': round(float(adj_r2), 3) if np.isfinite(adj_r2) else np.nan,
        })
    return pd.DataFrame(rows), fitted, clean


models = []
m1, _, _ = fit_ols(study, 'Log Innovation to Risk Ratio', ['AI Vibrancy Score'], 'H1: Framing ratio ~ AI vibrancy')
m2, _, _ = fit_ols(study, 'Log Innovation to Risk Ratio', ['IEF Overall Score'], 'H2: Framing ratio ~ IEF')
m3, combined_fitted, combined_data = fit_ols(
    study,
    'Log Innovation to Risk Ratio',
    ['AI Vibrancy Score', 'IEF Overall Score'],
    'H3 combined: Framing ratio ~ AI vibrancy + IEF',
)
models = pd.concat([m1, m2, m3], ignore_index=True)

model_path = outputs / f'correlation_study_regression_models_{FRAMING_SOURCE}.csv'
models.to_csv(model_path, index=False)
print(f'Saved: {model_path}')
models

## Mediation Diagnostic

This diagnostic follows the H3 mechanism: AI vibrancy influences framing permissiveness, and framing permissiveness is associated with economic freedom.

- X = AI Vibrancy Score
- M = Log Innovation to Risk Ratio
- Y = IEF Overall Score

With only eight countries, this is not a confirmatory mediation test. Treat it as a mechanism check.

In [ ]:
def ols_coefficients(df, y_col, x_cols):
    clean = df[[y_col, *x_cols]].dropna().copy()
    y = clean[y_col].to_numpy(dtype=float)
    x = clean[x_cols].to_numpy(dtype=float)
    x_design = np.column_stack([np.ones(len(clean)), x])
    beta, *_ = np.linalg.lstsq(x_design, y, rcond=None)
    return dict(zip(['Intercept', *x_cols], beta))


def coef_for(df, y_col, x_cols, term):
    return float(ols_coefficients(df, y_col, x_cols)[term])


mediation_df = study[['AI Vibrancy Score', 'Log Innovation to Risk Ratio', 'IEF Overall Score']].dropna().copy()
a = coef_for(mediation_df, 'Log Innovation to Risk Ratio', ['AI Vibrancy Score'], 'AI Vibrancy Score')
b = coef_for(mediation_df, 'IEF Overall Score', ['AI Vibrancy Score', 'Log Innovation to Risk Ratio'], 'Log Innovation to Risk Ratio')
c_total = coef_for(mediation_df, 'IEF Overall Score', ['AI Vibrancy Score'], 'AI Vibrancy Score')
c_prime = coef_for(mediation_df, 'IEF Overall Score', ['AI Vibrancy Score', 'Log Innovation to Risk Ratio'], 'AI Vibrancy Score')
indirect = a * b
mediated_share = indirect / c_total if c_total else np.nan

rng = np.random.default_rng(2026)
boot = []
for _ in range(1000):
    sample = mediation_df.sample(len(mediation_df), replace=True, random_state=int(rng.integers(0, 1_000_000_000)))
    try:
        a_i = coef_for(sample, 'Log Innovation to Risk Ratio', ['AI Vibrancy Score'], 'AI Vibrancy Score')
        b_i = coef_for(sample, 'IEF Overall Score', ['AI Vibrancy Score', 'Log Innovation to Risk Ratio'], 'Log Innovation to Risk Ratio')
        indirect_i = a_i * b_i
        if np.isfinite(indirect_i):
            boot.append(indirect_i)
    except Exception:
        continue

ci_low, ci_high = np.percentile(boot, [2.5, 97.5]) if boot else (np.nan, np.nan)
mediation = pd.DataFrame([{
    'N': len(mediation_df),
    'a: Framing ~ Vibrancy': round(a, 4),
    'b: IEF ~ Framing | Vibrancy': round(b, 4),
    'c total: IEF ~ Vibrancy': round(c_total, 4),
    "c': IEF ~ Vibrancy | Framing": round(c_prime, 4),
    'Indirect effect a*b': round(indirect, 4),
    'Bootstrap 95% CI low': round(float(ci_low), 4),
    'Bootstrap 95% CI high': round(float(ci_high), 4),
    'Mediated share': round(float(mediated_share), 4) if np.isfinite(mediated_share) else np.nan,
}])

mediation_path = outputs / f'correlation_study_mediation_{FRAMING_SOURCE}.csv'
mediation.to_csv(mediation_path, index=False)
print(f'Saved: {mediation_path}')
mediation

## Visualizations

In [ ]:
def scatter_with_fit(ax, df, x_col, y_col, title, xlabel, ylabel):
    clean = df[[x_col, y_col, 'Country']].dropna()
    ax.scatter(clean[x_col], clean[y_col], color='#1f77b4', s=60)
    slope, intercept, r, p, _ = stats.linregress(clean[x_col], clean[y_col])
    xs = np.linspace(clean[x_col].min(), clean[x_col].max(), 100)
    ax.plot(xs, intercept + slope * xs, color='#444', linewidth=1.5)
    for _, row in clean.iterrows():
        ax.annotate(row['Country'], (row[x_col], row[y_col]), xytext=(4, 4), textcoords='offset points', fontsize=8)
    ax.set_title(f'{title}\nr={r:.2f}, p={p:.3f}', fontsize=11)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)


fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
scatter_with_fit(
    axes[0], study,
    'AI Vibrancy Score', 'Log Innovation to Risk Ratio',
    'H1: Vibrancy vs framing ratio',
    'Stanford AI Vibrancy Score', 'Log innovation-to-risk framing ratio',
)
scatter_with_fit(
    axes[1], study,
    'IEF Overall Score', 'Log Innovation to Risk Ratio',
    'H2: IEF vs framing ratio',
    'IEF Overall Score', 'Log innovation-to-risk framing ratio',
)

axes[2].scatter(combined_fitted, combined_data['Log Innovation to Risk Ratio'], color='#2ca02c', s=60)
lims = [
    min(combined_fitted.min(), combined_data['Log Innovation to Risk Ratio'].min()),
    max(combined_fitted.max(), combined_data['Log Innovation to Risk Ratio'].max()),
]
axes[2].plot(lims, lims, color='#444', linewidth=1.5)
for country, x, y in zip(combined_data['Country'] if 'Country' in combined_data else study['Country'], combined_fitted, combined_data['Log Innovation to Risk Ratio']):
    axes[2].annotate(country, (x, y), xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[2].set_title('H3 combined model\nObserved vs fitted framing ratio', fontsize=11)
axes[2].set_xlabel('Fitted log ratio from Vibrancy + IEF')
axes[2].set_ylabel('Observed log innovation-to-risk ratio')
axes[2].grid(alpha=0.25)

fig.suptitle(f'Correlation Study ({FRAMING_SOURCE} framing)', fontsize=14, fontweight='bold')
fig.tight_layout()
figure_path = outputs / f'correlation_study_scatter_{FRAMING_SOURCE}.png'
fig.savefig(figure_path, dpi=200, bbox_inches='tight')
print(f'Saved: {figure_path}')
plt.show()

In [ ]:
matrix_cols = [
    'AI Vibrancy Score',
    'IEF Overall Score',
    'Innovation Framing per 1000 words',
    'Risk Framing per 1000 words',
    'Innovation to Risk Ratio',
    'Log Innovation to Risk Ratio',
]
corr_matrix = study[matrix_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(matrix_cols)))
ax.set_xticklabels(matrix_cols, rotation=45, ha='right')
ax.set_yticks(range(len(matrix_cols)))
ax.set_yticklabels(matrix_cols)
for i in range(len(matrix_cols)):
    for j in range(len(matrix_cols)):
        ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title(f'Pearson Correlation Matrix ({FRAMING_SOURCE} framing)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
matrix_path = outputs / f'correlation_study_matrix_{FRAMING_SOURCE}.png'
fig.savefig(matrix_path, dpi=200, bbox_inches='tight')
print(f'Saved: {matrix_path}')
plt.show()

## Interpretation Template

Use the coefficient direction and correlation strength rather than p-values alone, because the sample is only eight countries.

- H1 is supported directionally if AI Vibrancy is positively correlated with the log innovation-to-risk framing ratio.
- H2 is supported directionally if IEF is positively correlated with the log innovation-to-risk framing ratio, or if countries with higher framing ratios also have higher IEF scores.
- H3 is supported directionally if the combined model explains more variation in framing than either predictor alone, and the mediation diagnostic shows a plausible indirect path from Vibrancy to IEF through framing.